In [1]:
"""
Constructs the meshes from the volumetric data and from the 2.5D shapes.
"""

import napari_spatialdata.constants.config
import spatialdata as sd
from pathlib import Path
from numpy.random import default_rng

from tissue_map_tools.igneous_converters import (  # noqa: F401
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.data_model.annotations_utils import (
    make_dtypes_compatible_with_precomputed_annotations,
)
import time  # noqa: F401
import shutil  # noqa: F401
from tissue_map_tools.converters import (  # noqa: F401
    from_spatialdata_points_to_precomputed_points,
)
from tissue_map_tools.data_model.annotations_utils import parse_annotations

RNG = default_rng(42)

SMALL_DATA = True

/Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/.venv/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
out_path = Path.cwd() / "data"
sdata_zarr_path = out_path / "merfish_mouse_ileum.sdata.zarr"
precomputed_path = Path(str(out_path / "merfish_mouse_ileum_precomputed") + ('' if SMALL_DATA else '_full'))

# load the data
f = Path(sdata_zarr_path)
sdata = sd.read_zarr(f)

In [3]:
print(sd.get_extent(sdata["molecules"]))

{'x': (np.float64(112.0), np.float64(5720.0)), 'y': (np.float64(0.0), np.float64(9391.0)), 'z': (np.float64(0.0), np.float64(110.1455251))}


Let's subset the data in order to run this example notebook faster. Setting `SMALL_SDATA = False` will use the full data.

In [4]:
##
# subset the data
sdata_small = sd.bounding_box_query(
    sdata,
    axes=("x", "y", "z"),
    min_coordinate=[4000, 0, -10],
    max_coordinate=[5000, 1500, 200],
    target_coordinate_system="global",
)

/Users/macbook/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:909: UserWarning: The object has `points` element. Depending on the number of points, querying MAY suffer from performance issues. Please consider filtering the object before calling this function by calling the `subset()` method of `SpatialData`.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
# we need to transform the vector data to match the image due to this issue:
# https://github.com/hms-dbmi/tissue-map-tools/issues/13
transformation = sd.transformations.get_transformation(sdata_small["stains"])
translation_vector = transformation.to_affine_matrix(
    input_axes=("x", "y", "z"), output_axes=("x", "y", "z")
)[:3, 3]
translation = sd.transformations.Translation(translation_vector, axes=("x", "y", "z"))
for _, element_name, _ in sdata_small.gen_spatial_elements():
    old_transformation = sd.transformations.get_transformation(
        sdata_small[element_name]
    )
    sequence = sd.transformations.Sequence([old_transformation, translation.inverse()])
    sd.transformations.set_transformation(
        sdata_small[element_name],
        transformation=sequence,
        to_coordinate_system="global",
    )
    if sd.models.get_model(sdata_small[element_name]) not in (
        sd.models.Image3DModel,
        sd.models.Labels3DModel,
    ):
        transformed = sd.transform(sdata_small[element_name], to_coordinate_system="global")
        sdata_small[element_name] = transformed

if SMALL_DATA:
    sdata = sdata_small

Let's convert the `SpatialData` Zarr storage to the [Neuroglancer Precomputed format](https://github.com/google/neuroglancer/blob/master/src/datasource/precomputed/annotations.md), to enable visualization with `neuroglancer`.

In [6]:
from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
    raster=sdata["dapi_labels"],
    precomputed_path=str(precomputed_path),
)

Converted OME-Zarr data to the Precomputed format (segmentation) at /Users/macbook/embl/projects/basel/3d-spatial-workshop-2025/data/merfish_mouse_ileum_precomputed with pixel sizes {'x': 1000, 'y': 1000, 'z': 13768} and axes ['x', 'y', 'z'].
Volume Bounds:  Bbox([0, 0, 0],[1000, 1500, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[1000, 1500, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.44it/s]


Volume Bounds:  Bbox([0, 0, 0],[500, 750, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[500, 750, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 14.53it/s]


Volume Bounds:  Bbox([0, 0, 0],[250, 375, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[250, 375, 9], dtype=np.int32, unit='vx')


Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 32.56it/s]


Volume Bounds:  Bbox([0, 0, 0],[125, 188, 9], dtype=np.int32, unit='vx')
Selected ROI:   Bbox([0, 0, 0],[125, 188, 9], dtype=np.int32, unit='vx')


Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 2618.68it/s]

Tasks: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.97it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 2651.08it/s]

Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:00<00:00, 1362.49it/s]


We also create the meshes in the unsharded format (later we will use them in Vitessce, who only support the unsharded format so far).

In [7]:
from tissue_map_tools.igneous_converters import from_precomputed_raster_to_precomputed_meshes

from_precomputed_raster_to_precomputed_meshes(
    data_path=str(precomputed_path),
    mesh_name='mesh_mip_0_err_40_unsharded',
    sharded=False,
)

Tasks: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:01<00:00, 934.25it/s]


Here we choose if subsetting the points data or we keep the full dataset, and we choose which features to keep.

In [8]:
subset = RNG.choice(len(sdata["molecule_baysor"]), 10000, replace=False)

print(sdata["molecule_baysor"].columns)
if SMALL_DATA:
    subset_df = sdata["molecule_baysor"].compute().iloc[subset]
else:
    subset_df = sdata["molecule_baysor"].compute()
subset_df = subset_df[
    [
        "x",
        "y",
        "z",
        "gene",
        "area",
        "mol_id",
        "x_raw",
        "y_raw",
        "z_raw",
        "brightness",
        "total_magnitude",
        "compartment",
        "nuclei_probs",
        "assignment_confidence",
        "cell",
        "is_noise",
        "layer",
    ]
]
subset_df

Index(['mol_id', 'x_raw', 'y_raw', 'z_raw', 'gene', 'area', 'brightness',
       'total_magnitude', 'qc_score', 'molecule_id', 'confidence',
       'compartment', 'nuclei_probs', 'cell', 'assignment_confidence',
       'is_noise', 'ncv_color', 'layer', 'x', 'y', 'z'],
      dtype='object')


,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer
460642,500.0,841.0,34.420477,Lpar1,15,10095018,-2630.866,-1265.363,5.5,1.875414,1125.9140,Cyto,0.025976,0.675,867,False,3
423082,952.0,761.0,61.956858,Clca3b,6,10003055,-2581.609,-1274.128,8.5,2.150356,848.2183,Unknown,0.986717,1.000,1045,False,5
413076,84.0,444.0,6.884095,Txndc5,4,9988714,-2676.138,-1308.672,2.5,2.008227,407.6496,Unknown,0.967683,1.000,511,False,1
444843,715.0,1069.0,48.188667,Gp2,3,10048842,-2607.459,-1240.554,7.0,2.013750,309.6504,Unknown,0.940632,0.825,1085,False,4
451885,292.0,1144.0,6.884095,Sdc1,4,10069068,-2653.550,-1232.399,2.5,2.038763,437.3442,Unknown,0.818268,0.825,0,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426314,356.0,1094.0,6.884095,Adgrf5,15,10011807,-2646.565,-1237.818,2.5,2.164938,2192.9530,Unknown,1.000000,0.875,5284,False,1
426622,862.0,827.0,75.725049,Adgrf5,5,10012460,-2591.478,-1266.878,10.0,1.544077,175.0038,Cyto,0.019236,0.275,5702,False,6
442791,867.0,800.0,48.188667,Nlrp6,4,10044869,-2590.909,-1269.846,7.0,1.915653,329.3920,Cyto,0.091038,0.950,1022,False,4
450905,183.0,1297.0,34.420477,Mzb1,4,10067050,-2665.387,-1215.732,5.5,1.781510,241.8634,Unknown,1.000000,1.000,905,False,3


Ensure that the dtypes are compatible with the `neuroglancer` format. This will be made automatic in `tissue-map-tools`.

In [9]:
make_dtypes_compatible_with_precomputed_annotations(
    subset_df,
    max_categories=250,
    check_for_overflow=True,
)

,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer
460642,500.0,841.0,34.420475,Lpar1,15,10095018,-2630.865967,-1265.363037,5.5,1.875414,1125.913940,Cyto,0.025976,0.675,867,0,3
423082,952.0,761.0,61.956860,Clca3b,6,10003055,-2581.608887,-1274.128052,8.5,2.150356,848.218323,Unknown,0.986717,1.000,1045,0,5
413076,84.0,444.0,6.884095,Txndc5,4,9988714,-2676.137939,-1308.671997,2.5,2.008227,407.649597,Unknown,0.967683,1.000,511,0,1
444843,715.0,1069.0,48.188667,Gp2,3,10048842,-2607.458984,-1240.553955,7.0,2.013750,309.650391,Unknown,0.940632,0.825,1085,0,4
451885,292.0,1144.0,6.884095,Sdc1,4,10069068,-2653.550049,-1232.399048,2.5,2.038763,437.344208,Unknown,0.818268,0.825,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426314,356.0,1094.0,6.884095,Adgrf5,15,10011807,-2646.564941,-1237.817993,2.5,2.164938,2192.952881,Unknown,1.000000,0.875,5284,0,1
426622,862.0,827.0,75.725052,Adgrf5,5,10012460,-2591.478027,-1266.878052,10.0,1.544078,175.003799,Cyto,0.019236,0.275,5702,0,6
442791,867.0,800.0,48.188667,Nlrp6,4,10044869,-2590.908936,-1269.845947,7.0,1.915653,329.391998,Cyto,0.091038,0.950,1022,0,4
450905,183.0,1297.0,34.420475,Mzb1,4,10067050,-2665.386963,-1215.732056,5.5,1.781510,241.863403,Unknown,1.000000,1.000,905,0,3


In [10]:
sdata["molecule_baysor"] = sd.models.PointsModel.parse(subset_df)

# raster data converted to precomputed expresses units in nm therefore let's multiply the points by 1000
# this will be made more ergonomic as part of the tissue-map-tools APIs
for ax in ["x", "y", "z"]:
    sdata["molecule_baysor"][ax] = sdata["molecule_baysor"][ax] * 1000

print("converting the points to the precomputed format")

start = time.time()
path = Path(precomputed_path) / "molecule_baysor"
if path.exists():
    shutil.rmtree(path)

/Users/macbook/.local/share/uv/python/cpython-3.12.8-macos-aarch64-none/lib/python3.12/functools.py:946: UserWarning: The index of the dataframe is not monotonic increasing. It is recommended to sort the data to adjust the order of the index before calling .parse() (or call `parse(sort=True)`) to avoid possible problems due to unknown divisions.
  return method.__get__(obj, cls)(*args, **kwargs)


converting the points to the precomputed format


In [11]:
from_spatialdata_points_to_precomputed_points(
    sdata["molecule_baysor"],
    precomputed_path=precomputed_path,
    points_name="molecule_baysor",
    limit=10000,
)
print(f"conversion of points: {time.time() - start}")

Processing grid level 0 with shape (1, 1, 1) and chunk size [ 998000.        1498000.         110145.5234375]. Remaining points: 10000
Emitting 10000 points for grid cell (0, 0, 0)
{
    "type": "neuroglancer_annotations_v1",
    "dimensions": {
        "x": [
            1.0,
            "nm"
        ],
        "y": [
            1.0,
            "nm"
        ],
        "z": [
            1.0,
            "nm"
        ]
    },
    "lower_bound": [
        1000.0,
        1000.0,
        6884.09521484375
    ],
    "upper_bound": [
        999000.0,
        1499000.0,
        117029.6171875
    ],
    "annotation_type": "POINT",
    "properties": [
        {
            "id": "gene",
            "type": "int16",
            "description": "",
            "enum_values": [
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
        

In [12]:
print('done')

done


We have APIs to load the data from disk back to memory. Note the last columns `__spatial_index__` and `__chunk_key__`. These would enable compatibility with tools like [celldega](https://github.com/broadinstitute/celldega).

In [13]:
df_annotations = parse_annotations(Path(precomputed_path))
df_annotations

,x,y,z,gene,area,mol_id,x_raw,y_raw,z_raw,brightness,total_magnitude,compartment,nuclei_probs,assignment_confidence,cell,is_noise,layer,__spatial_index__,__chunk_key__
9458,784000.0,777000.0,20652.287109,Slc51a,9,9970083,-2599.879883,-1272.404053,4.0,1.838574,620.606201,Cyto,0.131446,1.000,904,0,2,spatial0,0_0_0
8215,856000.0,799000.0,117029.617188,Nlrp6,7,10047914,-2592.065918,-1269.985962,14.5,1.827326,470.353088,Cyto,0.001699,0.675,5261,0,9,spatial0,0_0_0
520,219000.0,178000.0,75725.054688,Cd44,6,10015483,-2661.492920,-1337.595947,10.0,1.859641,434.302490,Unknown,0.991415,0.875,454,0,6,spatial0,0_0_0
6603,803000.0,780000.0,117029.617188,Clca3b,8,10004899,-2597.836914,-1272.064941,14.5,1.914160,656.523010,Cyto,0.008495,0.625,5261,0,9,spatial0,0_0_0
4672,969000.0,519000.0,34420.476562,Maoa,4,9983797,-2579.771973,-1300.499023,5.5,2.186641,614.752991,Cyto,0.037952,0.525,895,0,3,spatial0,0_0_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4226,285000.0,641000.0,6884.095215,Slc51a,7,9967862,-2654.239990,-1287.209961,2.5,2.246747,1235.508057,Cyto,0.624436,0.600,5188,0,1,spatial0,0_0_0
4695,855000.0,684000.0,20652.287109,Maoa,6,9983115,-2592.137939,-1282.536011,4.0,1.669607,280.387207,Unknown,0.989746,1.000,941,0,2,spatial0,0_0_0
3574,336000.0,759000.0,20652.287109,Nlrp6,3,10043366,-2648.760010,-1274.355957,4.0,2.322277,630.083008,Cyto,0.196784,0.850,4979,0,2,spatial0,0_0_0
3627,283000.0,873000.0,117029.617188,Ifnar1,4,10061653,-2654.548096,-1261.937012,14.5,2.030568,429.168610,Cyto,0.107278,0.425,5166,0,9,spatial0,0_0_0
